# 00 · Setup and stores (run first)

Verifies the four validated source-graph stores the final framework reads (LCMV and wMNE, window modes B and A,
68 Desikan-Killiany ROIs) and records each one's content identity in the release lock
(`FINAL_v1/stores.lock.json`): from then on every experiment refuses a store whose content changed.
Nothing is built or modified. CPU runtime.

**How to use:** *Runtime → Run all*. Allow Google Drive access when asked. If Colab disconnects, open the notebook
again and *Run all* again: finished work is kept on Google Drive and skipped. Everything is written to
`<outputs>/FINAL_v1/` on Drive (never to the temporary Colab disk); the validated graph stores and all earlier
results are only read.

In [ ]:
# ===== Setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_CODE = False   # leave False. True only to move the release code pin deliberately to the newest commit
                      # (<outputs>/FINAL_v1/CODE_COMMIT.txt; unfinished work is then recomputed).
BRANCH = 'claude/unified-final-framework'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py'] + (['--update-code'] if UPDATE_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh
from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
FINAL = OUT + '/FINAL_v1'
print('release outputs on Drive:', FINAL)

## 1. Store verification (read-only)

In [ ]:
sh("python scripts/final.py stores")

## 2. Lock the stores and run the synthetic test suite once for this commit

In [ ]:
from adgraph.colab import run_tests_once
from adgraph.stores import open_store
for inverse in ('lcmv', 'wmne'):
    for mode in ('B', 'A'):
        store, rec = open_store(OUT, inverse, mode)
        print(f"{inverse} / mode {mode}: {rec['dir']} ({rec['n_windows']} windows, graph_run_id {rec['graph_run_id']})")
run_tests_once(OUT, '/content/repo')